# 02. Feature Engineering and WoE

This notebook creates simple dummy variables and coarse classes, then uses Weight of Evidence (WoE) and Information Value (IV) to inspect signal strength.

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split

DATA_PATH = Path('../data/loan_data_2007_2014.csv')
BAD_STATUSES = {
    'Charged Off', 'Default',
    'Does not meet the credit policy. Status:Charged Off',
    'Late (31-120 days)',
}

def load_and_clean():
    data = pd.read_csv(DATA_PATH, low_memory=False)
    required = ['loan_status', 'emp_length', 'term', 'earliest_cr_line', 'issue_d',
                'annual_inc', 'funded_amnt', 'total_rev_hi_lim', 'grade', 'int_rate', 'dti']
    missing = sorted(set(required) - set(data.columns))
    if missing:
        raise ValueError(f'Missing required columns: {missing}')
    data['good_bad'] = (~data['loan_status'].isin(BAD_STATUSES)).astype(int)
    data['emp_length_int'] = data['emp_length'].str.extract(r'(\d+)').astype(float).fillna(0).astype(int)
    data['term_int'] = data['term'].str.extract(r'(\d+)').astype(float).fillna(0).astype(int)
    for column in ['earliest_cr_line', 'issue_d']:
        dates = pd.to_datetime(data[column], format='%b-%y', errors='coerce')
        data[f'{column}_date'] = dates
        data[f'mths_since_{column}'] = ((pd.Timestamp('2017-12-01') - dates).dt.days / 30.4375).round()
    data['total_rev_hi_lim'] = data['total_rev_hi_lim'].fillna(data['funded_amnt'])
    data['annual_inc'] = data['annual_inc'].fillna(data['annual_inc'].median())
    for column in ['delinq_2yrs', 'inq_last_6mths', 'open_acc', 'pub_rec', 'total_acc', 'acc_now_delinq']:
        if column in data:
            data[column] = data[column].fillna(0)
    return data


In [ ]:
clean_data = load_and_clean()
feature_base = clean_data[['grade', 'home_ownership', 'verification_status', 'purpose', 'term_int',
                           'emp_length_int', 'int_rate', 'annual_inc', 'dti', 'good_bad']].copy()
feature_base['int_rate_band'] = pd.qcut(feature_base['int_rate'], 5, duplicates='drop')
feature_base['income_band'] = pd.qcut(feature_base['annual_inc'], 5, duplicates='drop')
feature_base['dti_band'] = pd.qcut(feature_base['dti'], 5, duplicates='drop')
engineered_data = pd.get_dummies(feature_base, columns=['grade', 'home_ownership', 'verification_status', 'purpose', 'int_rate_band', 'income_band', 'dti_band'], dtype=int)
engineered_data.shape

## Coarse classing and WoE / IV

The bins are deliberately coarse so each group is readable. WoE compares good and bad shares; IV is used here as a descriptive screening tool, not as proof of causality.

In [ ]:
def woe_iv(data, feature, target='good_bad'):
    table = data.groupby(feature, dropna=False)[target].agg(['count', 'sum'])
    table.columns = ['count', 'good']
    table['bad'] = table['count'] - table['good']
    table['good_share'] = (table['good'] + 0.5) / (table['good'].sum() + 0.5 * len(table))
    table['bad_share'] = (table['bad'] + 0.5) / (table['bad'].sum() + 0.5 * len(table))
    table['WoE'] = np.log(table['good_share'] / table['bad_share'])
    table['IV_component'] = (table['good_share'] - table['bad_share']) * table['WoE']
    table['IV'] = table['IV_component'].sum()
    return table.reset_index()

woe_grade = woe_iv(feature_base, 'grade')
woe_grade

In [ ]:
iv_summary = pd.DataFrame([
    {'feature': name, 'IV': woe_iv(feature_base, name)['IV'].iloc[0]}
    for name in ['grade', 'home_ownership', 'verification_status', 'purpose', 'term_int']
]).sort_values('IV', ascending=False)
iv_summary

In [ ]:
reference_categories = ['grade_G']
selected_features = [column for column in engineered_data.columns if column not in ['good_bad', *reference_categories]]
print(f'{len(selected_features)} selected dummy/coarse-class features')
selected_features[:20]

## Next stage

Notebook 03 uses these selected features with `grade_G` as a reference category, compares a broader and refined logistic specification, and models probability of good standing.